# Module 4: Linear Classifiers & Gradient Descent
## Case Study: Predictive Modeling for Public Water Safety

In this notebook we build three linear classifiers, from simplest to most robust:

1. **Perceptron** — a simple mistake-driven heuristic
2. **Gradient Descent (MSE loss)** — a global optimization approach
3. **Margin Classifier (Hinge Loss + L2 Regularization)** — a linear SVM-style model

Run each cell in order from top to bottom.

## 1. Data Acquisition & Cleaning

We load the water quality dataset and handle missing values before training. We also convert labels to `{-1, 1}` since our linear classifiers rely on that symmetry (more on this in the Discussion section below).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset from a public raw GitHub URL
url = "https://raw.githubusercontent.com/nferran/tp_aprendizaje_de_maquina_I/main/water_potability.csv"
df = pd.read_csv(url)

# Step 1: Handling Missing Values
# Water sensors often fail, leaving NaNs. We fill them with the mean of the column.
df.fillna(df.mean(), inplace=True)

# Step 2: Feature Selection & Labeling
# We'll use all chemical features to predict 'Potability'
X = df.drop('Potability', axis=1).values
y = df['Potability'].values

# Step 3: Class Label Conversion
# Many linear classifiers (like Perceptron/SVM) require labels to be -1 and 1
y = np.where(y == 0, -1, 1)

# Step 4: Train-Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset Loaded: {X_train.shape[0]} training samples, {X_train.shape[1]} features.")

**Why scaling matters:** gradient descent converges much faster and more reliably when features are on the same scale (mean 0, std 1). Without it, features with large numeric ranges dominate the gradient and slow down (or destabilize) training.

## 2. Phase 1: The Heuristic Approach (Perceptron)

The Perceptron doesn't have a "global" view of the error; it simply corrects itself every time it encounters a mistake.

**Intuition:** `y[i] * prediction <= 0` means the predicted sign disagrees with the true label (or is exactly zero, i.e. not confidently correct). When wrong, we move `w` in the direction of `y[i] * X[i]` — this rotates the decision boundary toward correctly classifying that point next time.

In [ ]:
class WaterPerceptron:
    def __init__(self, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.mistakes = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        for epoch in range(self.epochs):
            count = 0
            for i in range(len(y)):
                # Linear output: w . x + b
                prediction = np.dot(self.w, X[i]) + self.b

                # A "mistake" happens when the sign of the prediction
                # doesn't match the true label y[i].
                if y[i] * prediction <= 0:
                    self.w += self.lr * y[i] * X[i]
                    self.b += self.lr * y[i]
                    count += 1
            self.mistakes.append(count)

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

model_p = WaterPerceptron()
model_p.fit(X_train, y_train)
print("Perceptron trained.")

## 3. Phase 2: Gradient Descent - Global Optimization

The Perceptron is unstable if the data isn't perfectly separable. Here, instead of correcting one point at a time, we look at the **average** error across the whole dataset and take a small coordinated step downhill on that error surface (minimizing Mean Squared Error).

In [ ]:
class GDWaterClassifier:
    def __init__(self, lr=0.001, epochs=500):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0
        self.cost_history = []

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        n = X.shape[0]

        for _ in range(self.epochs):
            # 1. Linear output for ALL samples at once (vectorized)
            z = np.dot(X, self.w) + self.b

            # 2. Gradients of MSE loss w.r.t. w and b
            dw = (1 / n) * X.T.dot(z - y)
            db = (1 / n) * np.sum(z - y)

            # 3. Update parameters (step downhill)
            self.w -= self.lr * dw
            self.b -= self.lr * db

            # Track cost so we can plot convergence
            cost = (1 / n) * np.sum((z - y) ** 2)
            self.cost_history.append(cost)

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

model_gd = GDWaterClassifier()
model_gd.fit(X_train, y_train)
print("Gradient Descent model trained.")

**Why this differs from the Perceptron:** instead of updating after every single mistake, we compute the gradient of the *total* error across all samples, then take one coordinated step. This is why the loss curve is smooth — each step provably decreases the cost (assuming a reasonable learning rate), whereas the Perceptron just reacts locally and can bounce around.

## 4. Phase 3: Margin Classifiers & Hinge Loss

In water safety, we want more than just correctness — we want a **margin**, a safety gap between safe and unsafe samples. This is achieved using Hinge Loss combined with L2 Regularization:

$$\text{Loss} = \lambda \lVert w \rVert_2^2 + \sum_i \max(0, 1 - y_i(w^T x_i + b))$$

- **Hinge Loss**: `max(0, 1 - y_i*(w.x_i + b))` ensures correct classification *with a margin*.
- **L2 Regularization**: `lambda * ||w||^2` penalizes large weights, promoting generalization and stability.

**Intuition:** `y[i] * (w.x_i + b) >= 1` checks not just "is this point correct?" but "is it correct with at least margin 1?" If yes, we don't need to push the boundary toward it — we just shrink `w` slightly (regularization). If no (misclassified or too close to the boundary), we do a corrective update *plus* regularization.

In [ ]:
class MarginWaterClassifier:
    def __init__(self, lr=0.001, lambda_param=0.01, epochs=500):
        self.lr = lr
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.w = None
        self.b = 0

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        for _ in range(self.epochs):
            for i, x_i in enumerate(X):
                # Margin condition: is this point correctly classified
                # AND outside the margin (distance >= 1)?
                condition = y[i] * (np.dot(self.w, x_i) + self.b) >= 1

                if condition:
                    # Point is safely classified: only shrink w (regularization)
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Point violates the margin (or is misclassified):
                    # pull the boundary toward it AND regularize
                    self.w -= self.lr * (2 * self.lambda_param * self.w - x_i * y[i])
                    self.b -= self.lr * (-y[i])

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

model_margin = MarginWaterClassifier()
model_margin.fit(X_train, y_train)
print("Margin Classifier trained. (This cell may take longer — it loops sample by sample.)")

## 5. Critical Analysis & Comparison

### 5.1 Accuracy Report

In [ ]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

acc_p = accuracy(y_test, model_p.predict(X_test))
acc_gd = accuracy(y_test, model_gd.predict(X_test))
acc_margin = accuracy(y_test, model_margin.predict(X_test))

print(f"Perceptron Test Accuracy:        {acc_p:.4f}")
print(f"Gradient Descent Test Accuracy:  {acc_gd:.4f}")
print(f"Margin Classifier Test Accuracy: {acc_margin:.4f}")

### 5.2 Convergence Plot

Plot the mistakes history from Phase 1 (Perceptron) against the cost history from Phase 2 (Gradient Descent).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(model_p.mistakes)
axes[0].set_title("Perceptron: Mistakes per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Number of Mistakes")

axes[1].plot(model_gd.cost_history)
axes[1].set_title("Gradient Descent: Cost per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Cost")

plt.tight_layout()
plt.show()

**Why is the Gradient Descent plot smoother?** The Perceptron's "mistakes" count depends on the order of samples and only reacts to individual errors, so it can oscillate from epoch to epoch. Gradient Descent computes an *exact gradient over the entire dataset* at each step, so — with a sane learning rate — the cost decreases monotonically and smoothly.

### 5.3 Safety Margin

**Which model would you trust more for a sample close to the decision boundary — Perceptron or Margin Classifier?**

The **Margin Classifier**. The Perceptron only guarantees a point is on the *correct side* of the boundary — it stops updating the moment a point is classified correctly, even if that point sits right on the line. The Margin Classifier explicitly pushes points to be at least a distance of 1 away from the boundary (via the hinge loss condition `y*(w.x+b) >= 1`), so predictions near the boundary are inherently rarer and, when they do occur, the model has actively tried to keep genuinely uncertain points from sitting exactly at the edge. For a safety-critical application like water potability, that extra buffer is exactly what you want.

## 6. Discussion Questions

### Q1: Impact of High Learning Rate in Gradient Descent

**What happens if `learning_rate` is set too high (e.g., 1.0)?**

The update step `w -= lr * dw` becomes too large. Instead of taking a small, careful step downhill on the cost surface, the model **overshoots** the minimum. This can cause the loss to oscillate wildly or even **diverge** (grow toward infinity / become `NaN`) instead of converging. Try it yourself in the cell below.

### Q2: Label Conversion in Classification

**Why convert labels to {-1, 1} instead of keeping {0, 1}?**

With `{-1, 1}` labels, the expression `y * (w.x + b)` has a clean geometric meaning: positive means "correctly classified," negative means "misclassified," and its magnitude is proportional to the distance from the boundary. This symmetry is essential for Hinge Loss (`max(0, 1 - y*(w.x+b))`) and the Perceptron's mistake condition (`y*prediction <= 0`) to work correctly. With `{0, 1}` labels, this clean symmetric relationship breaks down and the same formulas wouldn't behave consistently for both classes.

### Q3: Handling Noisy Data

**Which algorithm best handles noisy, non-separable data like the Water Potability dataset?**

The **Margin Classifier (Hinge Loss + L2 Regularization)** is generally the most robust:
- The **Perceptron** has no concept of "how wrong" a point is, and on non-separable data it can keep flipping the boundary back and forth forever without converging.
- Plain **Gradient Descent with MSE loss** penalizes errors *quadratically*, making it overly sensitive to outliers far from the boundary.
- The **hinge loss** only penalizes points that violate the margin, and does so *linearly* rather than quadratically, while the **L2 term** discourages the boundary from overfitting to noisy points — giving both robustness and better generalization.

### (Optional) Experiment: try a very high learning rate

Run this to see divergence in action for Q1.

In [ ]:
model_gd_high_lr = GDWaterClassifier(lr=1.0, epochs=50)
model_gd_high_lr.fit(X_train, y_train)

plt.plot(model_gd_high_lr.cost_history)
plt.title("Gradient Descent with lr=1.0 (Divergence)")
plt.xlabel("Epoch")
plt.ylabel("MSE Cost")
plt.show()

print("Final cost:", model_gd_high_lr.cost_history[-1])